# 03 — Silver: Enrich & Transform
**Bronze → Silver medallion transformation**

Reads the station snapshot from **bikerental_bronze_raw** (landed by Eventstream `RTIbikeRental`)
and produces five enriched Silver tables in **bicycles_silver**.

### Data Reality
The BicyclesRental sample data is a **point-in-time snapshot** of ~195 stations
(7 columns, no timestamps). It is NOT a continuous time-series stream.
This notebook:
- Enriches what we **have** (spatial, capacity, utilization analytics)
- Generates a realistic 24-hour demand profile per neighbourhood so the
  ML notebook (NB06) and Gold star schema (NB04) have temporal data to work with

| Silver Table | Description |
|---|---|
| `silver_station_profile` | Station master — capacity, utilization (0–1), rebalance priority |
| `silver_availability_events` | Station snapshot + availability classification + criticality |
| `silver_neighbourhood_metrics` | Neighbourhood-level aggregated demand & health scores |
| `silver_rebalancing_candidates` | Stations flagged for fleet rebalancing with recommended actions |
| `silver_hourly_demand` | Generated 24 h × neighbourhood demand profile for ML forecasting |

### Prerequisites
1. Attach **bicycles_silver** as the default lakehouse
2. Add **bikerental_bronze_raw** as an additional lakehouse
3. Eventstream `RTIbikeRental` should have run at least once

### Utilization Scale: **0 – 1** everywhere
`utilization_pct = No_Bikes / (No_Bikes + No_Empty_Docks)` — Power BI formats as percentage.

In [ ]:
# ============================================================
# CELL 1 — Configuration · Imports · Read Bronze
# ============================================================

from pyspark.sql.functions import (
    col, lit, when, coalesce, round as spark_round,
    count, countDistinct, avg as spark_avg, sum as spark_sum,
    min as spark_min, max as spark_max, stddev,
    hour, dayofweek, date_format, date_trunc, to_date,
    row_number, current_timestamp, monotonically_increasing_id,
    concat_ws, sha2, expr, explode, array, struct,
    rand, abs as spark_abs
)
from pyspark.sql.window import Window
from pyspark.sql.types import (
    IntegerType, DoubleType, LongType, StringType,
    StructType, StructField, TimestampType,
)
from pyspark.sql import SparkSession
from datetime import datetime, timedelta

spark = SparkSession.builder.getOrCreate()

# ── Configuration — use fully qualified table names for pipeline execution ──
# In Fabric, schema-enabled lakehouses: {LAKEHOUSE}.dbo.{table}
# Non-schema lakehouses:                {LAKEHOUSE}.{table}
BRONZE_LAKEHOUSE = "bikerental_bronze_raw"
BRONZE_SCHEMA    = f"{BRONZE_LAKEHOUSE}.dbo"       # Tables under dbo schema

SILVER_LAKEHOUSE = "bicycles_silver"
SILVER_SCHEMA    = f"{SILVER_LAKEHOUSE}"            # No dbo schema

# ABFSS fallback IDs (if spark_catalog doesn't resolve .dbo)
WORKSPACE_ID        = "573cc7c7-a45a-4fd9-886e-9db4e9798db4"
BRONZE_LAKEHOUSE_ID = "890af2be-6762-5333-8e9f-f44bd626c040"

print(f"Spark version: {spark.version}")
print(f"Processing started at: {datetime.now()}")
print(f"\nSource: {BRONZE_SCHEMA}")
print(f"Target: {SILVER_SCHEMA}")

# ── Verify Bronze tables ──────────────────────────────────────
print(f"\nBronze tables ({BRONZE_SCHEMA}):")
try:
    spark.sql(f"SHOW TABLES IN {BRONZE_SCHEMA}").show(truncate=False)
except Exception as e:
    print(f"  Could not list tables via catalog: {e}")

# ── Read bronze ───────────────────────────────────────────────
try:
    df_raw = spark.sql(f"SELECT * FROM {BRONZE_SCHEMA}.bikeraw_tb")
    print(f"✓ Reading bronze via: {BRONZE_SCHEMA}.bikeraw_tb")
except Exception:
    # Fallback: direct Delta read via ABFSS (schema-enabled lakehouse)
    _abfss = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{BRONZE_LAKEHOUSE_ID}"
    df_raw = spark.read.format("delta").load(f"{_abfss}/Tables/dbo/bikeraw_tb")
    print(f"✓ Reading bronze via ABFSS fallback: bikerental_bronze_raw/dbo/bikeraw_tb")

ROW_COUNT = df_raw.count()
print(f"Bronze columns : {df_raw.columns}")
print(f"Bronze rows    : {ROW_COUNT:,}")

# ── Add foundational columns ─────────────────────────────────
df_raw = (
    df_raw
    .withColumn("No_Bikes",       col("No_Bikes").cast(IntegerType()))
    .withColumn("No_Empty_Docks", col("No_Empty_Docks").cast(IntegerType()))
    .withColumn("Latitude",       col("Latitude").cast(DoubleType()))
    .withColumn("Longitude",      col("Longitude").cast(DoubleType()))
    .withColumn("total_docks",    col("No_Bikes") + col("No_Empty_Docks"))
    .withColumn(
        "utilization_pct",
        when(
            (col("No_Bikes") + col("No_Empty_Docks")) > 0,
            spark_round(col("No_Bikes") / (col("No_Bikes") + col("No_Empty_Docks")), 4),
        ).otherwise(lit(0.0)),
    )
    .withColumn("snapshot_time", current_timestamp())
)

print(f"Enriched rows  : {df_raw.count():,}")
print(f"Columns        : {df_raw.columns}")

# ── Incremental Load Utilities ────────────────────────────────
# First run  → Full overwrite (tables don't exist yet)
# Schema upgrade → Old table has missing columns → overwrite to fix
# Incremental → Watermark on source_timestamp → only new records
# Merge fallback → No Eventstream timestamp → MERGE dedup
# ──────────────────────────────────────────────────────────────

def table_exists(fqn):
    """Check if a Delta table already exists in the lakehouse."""
    try:
        spark.sql(f"DESCRIBE TABLE {fqn}")
        return True
    except Exception:
        return False

def column_exists(fqn, col_name):
    """Check if a specific column exists in an existing table."""
    if not table_exists(fqn):
        return False
    try:
        schema = spark.sql(f"SELECT * FROM {fqn} LIMIT 0").schema
        return col_name in [f.name for f in schema.fields]
    except Exception:
        return False

def get_max_value(fqn, col_name):
    """Return MAX(col_name), or None if table/column doesn't exist."""
    if not table_exists(fqn):
        return None
    if not column_exists(fqn, col_name):
        return None
    try:
        row = spark.sql(f"SELECT MAX(`{col_name}`) AS wm FROM {fqn}").collect()[0]
        return row["wm"]
    except Exception:
        return None

# ── Detect Eventstream system timestamp in bronze ─────────────
_ES_TS_CANDIDATES = ["EventProcessedUtcTime", "EventEnqueuedUtcTime", "_timestamp", "timestamp"]
BRONZE_TS_COL = next((c for c in df_raw.columns if c in _ES_TS_CANDIDATES), None)

if BRONZE_TS_COL:
    print(f"\n✓ Eventstream timestamp detected: {BRONZE_TS_COL}")
    print("  Incremental loads will filter on this column")
else:
    print("\nℹ No Eventstream timestamp in bronze (static sample data)")

# ── Determine run mode ────────────────────────────────────────
# Decision tree:
#   1. Table doesn't exist?           → FULL (first run)
#   2. Table missing source_timestamp? → FULL (schema upgrade)
#   3. Eventstream timestamp exists?   → INCREMENTAL (watermark filter)
#   4. Otherwise                       → FULL overwrite (no way to filter)
_events_target = f"{SILVER_LAKEHOUSE}.silver_availability_events"
_table_found   = table_exists(_events_target)
_has_source_ts = column_exists(_events_target, "source_timestamp") if _table_found else False

if not _table_found:
    # Scenario A: first run ever
    LOAD_MODE = "full"
    df_raw_incremental = df_raw
    print("Run mode: FIRST RUN — full load (tables will be created)")

elif not _has_source_ts:
    # Scenario B/C: old table from previous code, missing source_timestamp
    # Must overwrite to establish new schema
    LOAD_MODE = "full"
    df_raw_incremental = df_raw
    print("Run mode: SCHEMA UPGRADE — overwriting Silver to add source_timestamp column")

elif BRONZE_TS_COL:
    # Scenario D: proper incremental — both column and timestamp exist
    _watermark = get_max_value(_events_target, "source_timestamp")
    if _watermark:
        df_raw_incremental = df_raw.filter(col(BRONZE_TS_COL) > lit(_watermark))
        _incr_count = df_raw_incremental.count()
        if _incr_count > 0:
            LOAD_MODE = "incremental"
            print(f"Run mode: INCREMENTAL — {_incr_count:,} new bronze rows since {_watermark}")
        else:
            LOAD_MODE = "skip"
            print(f"Run mode: SKIP — no new bronze data since {_watermark}")
    else:
        # Table has source_timestamp column but values are all NULL (transition run)
        df_raw_incremental = df_raw
        LOAD_MODE = "full"
        print("Run mode: FULL — source_timestamp has no values yet (transition run)")
else:
    # Scenario E: no Eventstream timestamp, table exists with schema
    # Full overwrite — can't filter without a timestamp column in bronze
    LOAD_MODE = "full"
    df_raw_incremental = df_raw
    print("Run mode: FULL REFRESH — no Eventstream timestamp to filter on")


In [ ]:
# ============================================================
# CELL 2 — silver_station_profile (Station Master · Deduped)
# ============================================================
# Bronze accumulates many Eventstream snapshots over time.
# Each snapshot repeats all ~800 stations with updated bike counts.
# dropDuplicates(["BikepointID"]) guarantees exactly ONE row per station.
# This is simpler and more memory-efficient than Window + row_number
# for large datasets (561M+ rows).

_pre_dedup = df_raw.select("BikepointID").distinct().count()
print(f"Unique BikepointIDs in bronze: {_pre_dedup}")

df_station_profile = (
    df_raw
    .dropDuplicates(["BikepointID"])
    .select(
        "BikepointID", "Street", "Neighbourhood",
        "Latitude", "Longitude",
        "No_Bikes", "No_Empty_Docks", "total_docks",
        "utilization_pct",
        col("snapshot_time").alias("last_seen_at"),
    )
    .withColumn(
        "station_size",
        when(col("total_docks") >= 40, "large")
        .when(col("total_docks") >= 20, "medium")
        .otherwise("small"),
    )
    .withColumn(
        "availability_status",
        when(col("No_Bikes") == 0, "empty")
        .when(col("No_Empty_Docks") == 0, "full")
        .when(col("utilization_pct") < 0.20, "low")
        .when(col("utilization_pct") > 0.80, "high")
        .otherwise("normal"),
    )
    .withColumn(
        "rebalance_priority",
        spark_round(
            when(col("No_Bikes") == 0, lit(100))
            .when(col("No_Empty_Docks") == 0, lit(90))
            .when(col("utilization_pct") < 0.10, lit(80))
            .when(col("utilization_pct") > 0.90, lit(75))
            .when(col("utilization_pct") < 0.20, lit(50))
            .when(col("utilization_pct") > 0.80, lit(45))
            .otherwise(lit(0)),
            0,
        ),
    )
    .withColumn("processed_at", current_timestamp())
)

# Safety check — station count should be ~800, NOT 259K
_cnt_check = df_station_profile.count()
print(f"Station profile rows after dedup: {_cnt_check}")
assert _cnt_check < 2000, (
    f"DEDUP FAILED: station_profile has {_cnt_check:,} rows "
    f"(expected ~800). Check BikepointID uniqueness."
)

df_station_profile.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{SILVER_LAKEHOUSE}.silver_station_profile")

_cnt = spark.sql(
    f"SELECT COUNT(*) AS c FROM {SILVER_LAKEHOUSE}.silver_station_profile"
).collect()[0]["c"]
print(f"✓ silver_station_profile: {_cnt} stations (deduped to 1 per BikepointID)")
df_station_profile.groupBy("availability_status").count() \
    .orderBy("count", ascending=False).show()


In [ ]:
# ============================================================
# CELL 3 — silver_availability_events (INCREMENTAL)
# ============================================================
# Each station row becomes an availability event.
# Enriched with availability classification and criticality flags.
#
# Incremental strategy:
#   full        → Overwrite table (first run, schema upgrade, or no timestamp)
#   incremental → Append only new rows (watermark-filtered)
#   skip        → No new data, do nothing

# ── Build timestamp-aware hash components ─────────────────────
_ts_col = col(BRONZE_TS_COL) if BRONZE_TS_COL else col("snapshot_time")
_hash_cols = [
    col("BikepointID"),
    col("No_Bikes").cast("string"),
    col("No_Empty_Docks").cast("string"),
]
if BRONZE_TS_COL:
    _hash_cols.append(col(BRONZE_TS_COL).cast("string"))
else:
    _hash_cols.append(col("snapshot_time").cast("string"))

if LOAD_MODE != "skip":
    df_events = (
        df_raw_incremental
        .withColumn(
            "event_id",
            sha2(concat_ws("|", *_hash_cols), 256),
        )
        .withColumn("event_timestamp", _ts_col)
        .withColumn("source_timestamp", _ts_col)
        .withColumn(
            "availability_band",
            when(col("utilization_pct") == 0, "empty")
            .when(col("utilization_pct") < 0.20, "critical_low")
            .when(col("utilization_pct") < 0.40, "low")
            .when(col("utilization_pct") < 0.60, "balanced")
            .when(col("utilization_pct") < 0.80, "high")
            .when(col("utilization_pct") < 1.00, "near_full")
            .otherwise("full"),
        )
        .withColumn(
            "event_type",
            when(col("No_Bikes") == 0, "empty_station")
            .when(col("No_Empty_Docks") == 0, "full_station")
            .when(col("utilization_pct") < 0.15, "rebalance_needed")
            .when(col("utilization_pct") > 0.85, "demand_spike")
            .otherwise("normal"),
        )
        .withColumn(
            "is_critical",
            when(
                (col("No_Bikes") == 0) | (col("No_Empty_Docks") == 0)
                | (col("utilization_pct") < 0.10) | (col("utilization_pct") > 0.90),
                True,
            ).otherwise(False),
        )
        .select(
            "event_id", "event_timestamp", "source_timestamp",
            "BikepointID", "Street",
            "Neighbourhood", "No_Bikes", "No_Empty_Docks",
            "total_docks", "utilization_pct",
            "event_type", "is_critical", "availability_band",
        )
        .withColumn("processed_at", current_timestamp())
    )

# ── Write based on LOAD_MODE ─────────────────────────────────
_target = f"{SILVER_LAKEHOUSE}.silver_availability_events"

if LOAD_MODE == "full":
    df_events.write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true").saveAsTable(_target)
    _write_status = "OVERWRITTEN (full refresh)"
elif LOAD_MODE == "incremental":
    _new_count = df_events.count()
    if _new_count > 0:
        df_events.write.format("delta").mode("append").saveAsTable(_target)
        _write_status = f"APPENDED {_new_count:,} new events"
    else:
        _write_status = "SKIPPED — 0 new rows after filter"
elif LOAD_MODE == "skip":
    _write_status = "SKIPPED — no new data since last watermark"
else:
    _write_status = f"UNKNOWN LOAD_MODE: {LOAD_MODE}"

_cnt = spark.sql(f"SELECT COUNT(*) AS c FROM {_target}").collect()[0]["c"]
print(f"✓ silver_availability_events: {_cnt:,} events [{_write_status}]")
if LOAD_MODE != "skip":
    df_events.groupBy("event_type").count().orderBy("count", ascending=False).show()


In [ ]:
# ============================================================
# CELL 4 — silver_neighbourhood_metrics
# ============================================================
# Aggregated neighbourhood-level capacity, health score, and demand.
# Derived from silver_station_profile (utilization 0–1 scale).

df_stations = spark.sql(f"SELECT * FROM {SILVER_LAKEHOUSE}.silver_station_profile")

df_neighbourhood = (
    df_stations.groupBy("Neighbourhood")
    .agg(
        count("*").alias("station_count"),
        spark_sum("total_docks").alias("total_capacity"),
        spark_sum("No_Bikes").alias("total_bikes_available"),
        spark_sum("No_Empty_Docks").alias("total_empty_docks"),
        spark_avg("utilization_pct").alias("avg_utilization_pct"),
        spark_min("utilization_pct").alias("min_utilization_pct"),
        spark_max("utilization_pct").alias("max_utilization_pct"),
        stddev("utilization_pct").alias("std_utilization_pct"),
        spark_avg("Latitude").alias("centroid_lat"),
        spark_avg("Longitude").alias("centroid_lon"),
        spark_sum(when(col("availability_status") == "empty", 1).otherwise(0)).alias("empty_stations"),
        spark_sum(when(col("availability_status") == "full", 1).otherwise(0)).alias("full_stations"),
        spark_sum(when(col("rebalance_priority") > 0, 1).otherwise(0)).alias("stations_needing_rebalance"),
    )
    .withColumn(
        "neighbourhood_utilization_pct",
        when(
            col("total_capacity") > 0,
            spark_round(col("total_bikes_available").cast(DoubleType()) / col("total_capacity"), 4),
        ).otherwise(lit(0.0)),
    )
    .withColumn(
        "health_score",
        spark_round(
            lit(100)
            - (col("empty_stations") / col("station_count") * 40)
            - (col("full_stations") / col("station_count") * 30)
            - (coalesce(col("std_utilization_pct"), lit(0)) / lit(0.5) * 30),
            1,
        ),
    )
    .withColumn(
        "capacity_status",
        when(col("neighbourhood_utilization_pct") < 0.20, "under_utilized")
        .when(col("neighbourhood_utilization_pct") > 0.80, "over_utilized")
        .otherwise("balanced"),
    )
    .withColumn("processed_at", current_timestamp())
)

df_neighbourhood.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable(f"{SILVER_LAKEHOUSE}.silver_neighbourhood_metrics")

_cnt = spark.sql(f"SELECT COUNT(*) AS c FROM {SILVER_LAKEHOUSE}.silver_neighbourhood_metrics").collect()[0]["c"]
print(f"✓ silver_neighbourhood_metrics: {_cnt} neighbourhoods")
df_neighbourhood.select(
    "Neighbourhood", "station_count", "total_capacity",
    "neighbourhood_utilization_pct", "health_score", "capacity_status",
).orderBy("health_score", ascending=False).show(10, truncate=False)

In [ ]:
# ============================================================
# CELL 5 — silver_rebalancing_candidates
# ============================================================
# Stations where rebalance_priority > 0, with recommended actions
# and bikes-to-target calculations.  utilization_pct is 0–1 scale.

df_stations = spark.sql(f"SELECT * FROM {SILVER_LAKEHOUSE}.silver_station_profile")

df_rebalance = (
    df_stations
    .filter(col("rebalance_priority") > 0)
    .select(
        "BikepointID", "Street", "Neighbourhood",
        "Latitude", "Longitude",
        "No_Bikes", "No_Empty_Docks", "total_docks",
        "utilization_pct", "availability_status",
        "rebalance_priority", "station_size",
    )
    .withColumn(
        "recommended_action",
        when(col("availability_status") == "empty", "URGENT: Deploy bikes")
        .when(col("availability_status") == "full", "URGENT: Remove bikes")
        .when(col("utilization_pct") < 0.20, "Add bikes (low availability)")
        .when(col("utilization_pct") > 0.80, "Remove bikes (near capacity)")
        .otherwise("Monitor"),
    )
    .withColumn(
        "bikes_to_target",
        spark_round(col("total_docks") * 0.5 - col("No_Bikes"), 0).cast(IntegerType()),
    )
    .withColumn("processed_at", current_timestamp())
    .orderBy(col("rebalance_priority").desc())
)

df_rebalance.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable(f"{SILVER_LAKEHOUSE}.silver_rebalancing_candidates")

_cnt = spark.sql(f"SELECT COUNT(*) AS c FROM {SILVER_LAKEHOUSE}.silver_rebalancing_candidates").collect()[0]["c"]
print(f"✓ silver_rebalancing_candidates: {_cnt} stations need rebalancing")
display(df_rebalance.limit(15))

In [ ]:
# ============================================================
# CELL 6 — silver_hourly_demand (Generated 24 h Profile) + Summary
# ============================================================
# BicyclesRental sample data is a single snapshot — no time series.
# We generate a realistic 24-hour × 7-day demand profile per
# neighbourhood based on observed station utilization + TfL-style
# demand curves. This feeds NB06 ML and NB04 Gold star schema.

import math
from pyspark.sql import Row

# ── TfL-style demand multipliers by hour (London bike-share) ──
# Values represent relative demand (1.0 = baseline)
WEEKDAY_CURVE = {
    0: 0.10, 1: 0.05, 2: 0.03, 3: 0.02, 4: 0.03, 5: 0.08,
    6: 0.25, 7: 0.70, 8: 1.00, 9: 0.85, 10: 0.55, 11: 0.50,
    12: 0.60, 13: 0.55, 14: 0.50, 15: 0.55, 16: 0.65, 17: 0.95,
    18: 1.00, 19: 0.75, 20: 0.45, 21: 0.30, 22: 0.20, 23: 0.12,
}
WEEKEND_CURVE = {
    0: 0.08, 1: 0.05, 2: 0.03, 3: 0.02, 4: 0.02, 5: 0.04,
    6: 0.10, 7: 0.15, 8: 0.30, 9: 0.50, 10: 0.70, 11: 0.85,
    12: 0.95, 13: 1.00, 14: 0.95, 15: 0.90, 16: 0.80, 17: 0.70,
    18: 0.55, 19: 0.40, 20: 0.30, 21: 0.20, 22: 0.15, 23: 0.10,
}

# ── Get neighbourhood profiles from station data ─────────────
df_stations = spark.sql(f"SELECT * FROM {SILVER_LAKEHOUSE}.silver_station_profile")

hood_agg = (
    df_stations.groupBy("Neighbourhood")
    .agg(
        count("*").alias("station_count"),
        spark_sum("total_docks").alias("total_capacity"),
        spark_avg("utilization_pct").alias("base_utilization"),
    )
    .collect()
)

# ── Generate 24 h × 7 days × neighbourhood rows ──────────────
from datetime import datetime, timedelta

base_date = datetime.now().replace(hour=0, minute=0, second=0, microsecond=0)
rows = []

for hood_row in hood_agg:
    hood = hood_row["Neighbourhood"]
    n_stations = hood_row["station_count"]
    capacity = hood_row["total_capacity"]
    base_util = float(hood_row["base_utilization"]) if hood_row["base_utilization"] else 0.5

    for day_offset in range(7):  # 7 days
        dt = base_date + timedelta(days=day_offset)
        dow = dt.weekday()  # 0=Mon..6=Sun
        is_weekend = dow >= 5
        curve = WEEKEND_CURVE if is_weekend else WEEKDAY_CURVE

        for h in range(24):
            event_hour = dt + timedelta(hours=h)
            multiplier = curve[h]

            # Demand pressure = multiplier × base utilization × station density
            demand = multiplier * base_util * (capacity / 20.0)
            # Add slight deterministic variation per neighbourhood
            seed = hash(f"{hood}_{day_offset}_{h}") % 100 / 100.0
            demand = demand * (0.9 + seed * 0.2)  # ±10% variation

            avg_util = min(base_util * multiplier * (0.95 + seed * 0.1), 1.0)
            avg_bikes = avg_util * (capacity / n_stations) if n_stations > 0 else 0

            is_rush = h in (7, 8, 9, 17, 18, 19)
            critical = 1 if (avg_util > 0.9 or avg_util < 0.1) else 0
            rebalance_triggers = 1 if avg_util < 0.15 else 0
            spike_count = 1 if avg_util > 0.85 else 0

            rows.append(Row(
                event_hour=event_hour,
                Neighbourhood=hood,
                event_count=n_stations,  # each station contributes 1 observation
                avg_utilization=round(avg_util, 4),
                min_bikes=max(0, int(avg_bikes * 0.5)),
                max_bikes=int(min(avg_bikes * 1.5, capacity / max(n_stations, 1))),
                avg_bikes=round(avg_bikes, 1),
                critical_events=critical,
                rebalance_triggers=rebalance_triggers,
                demand_spike_count=spike_count,
                active_stations=n_stations,
                hour_of_day=h,
                day_of_week=dow + 1,  # Spark convention: 1=Sun..7=Sat → shift
                is_rush_hour=is_rush,
            ))

df_hourly = spark.createDataFrame(rows)
df_hourly = df_hourly.withColumn("processed_at", current_timestamp())

df_hourly.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable(f"{SILVER_LAKEHOUSE}.silver_hourly_demand")

_cnt = spark.sql(f"SELECT COUNT(*) AS c FROM {SILVER_LAKEHOUSE}.silver_hourly_demand").collect()[0]["c"]
_hoods = df_hourly.select("Neighbourhood").distinct().count()
print(f"✓ silver_hourly_demand: {_cnt:,} rows  ({_hoods} neighbourhoods × 7 days × 24 hours)")

# ── Summary ──────────────────────────────────────────────────
print("\n" + "=" * 60)
print("SILVER LAYER COMPLETE")
print("=" * 60)

tables = {
    "silver_station_profile":        "Station master (util 0–1)",
    "silver_availability_events":    "Availability snapshot (util 0–1)",
    "silver_neighbourhood_metrics":  "Neighbourhood aggregation",
    "silver_rebalancing_candidates": "Rebalance targets",
    "silver_hourly_demand":          "24h × 7d demand profile for ML",
}
print(f"\n{'Table':<35} {'Rows':>10}  Description")
print("-" * 70)
for tbl, desc in tables.items():
    try:
        n = spark.sql(f"SELECT COUNT(*) AS c FROM {SILVER_LAKEHOUSE}.{tbl}").collect()[0]["c"]
        print(f"  {tbl:<33} {n:>10,}  {desc}")
    except Exception:
        print(f"  {tbl:<33} {'MISSING':>10}  {desc}")

print("\nNEXT → Run NB04 (Gold Star Schema)")